In [1]:
import pandas as pd
import numpy as np

# 1. Load Raw Dataset

df = pd.read_csv("response.csv")
print(df.columns.tolist())


['Timestamp', 'Name of the Student', 'University Roll No.', 'Class Roll No', 'Year of Study', 'Age ', 'Course', 'Department', 'Have you taken a MOOC course before?', 'Which MOOC platform did you use?', 'What was the domain of your course?', 'Course Duration', 'Course Start Date', 'Course End Date', 'Course Difficulty Level (self rated)', 'Was the course self placed or instructor paced?', 'Did your course contain more visual effects for better understanding of concepts?', 'Average study time per week', 'How many lectures did you complete?', 'How many assignments did you complete?', 'Did you participate in discussion forums?', 'Did you take notes while learning?', 'How often did you revisit previous content?', 'Did you take all the quiz and tests?', 'How many total exams you had to take in order to pass the course?', 'Were the questions in exams similar to assignments and class or were they significantly different? ', 'Was your final exam conducted in online or offline manner?', 'Why did

In [2]:


# 2. Clean Column Names (strip spaces)
df.columns = df.columns.str.strip()


In [3]:

# 3. Convert all text values to lowercase & strip

def clean_text(x):
    if isinstance(x, str):
        return x.strip().lower()
    return x

df = df.applymap(clean_text)

C:\Users\ishika\AppData\Local\Temp\ipykernel_11432\1091572817.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_text)


In [4]:
# Clean column names (VERY IMPORTANT FIX)
df.columns = df.columns.str.strip().str.lower()

# Create dropout target
drop_conditions = (
    df['final result status'].str.contains("withdraw", na=False) |
    df['did you complete the course?'].str.contains("no", na=False) |
    df['did you pass ,fail or withdraw?'].str.contains("withdraw", na=False)
)

df['dropout'] = np.where(drop_conditions, 1, 0)
# Create marks range target (for marks model)
df['marks_range'] = df['your exam score(approx)']



In [5]:
# 5. Remove Irrelevant Columns

irrelevant_cols = [
    'Timestamp',
    'Name of the Student',
    'University Roll No.',
    'Class Roll No',
    'Year of Study',
    'Age ',
    'Course Start Date',
    'Course End Date'
]

df.drop(columns=[col for col in irrelevant_cols if col in df.columns], inplace=True)


In [6]:

# 6. Remove Leakage Columns (TASK-WISE)

# ---- Leakage for DROPOUT model (post-outcome info) ----
dropout_leakage_cols = [
    'did you complete the course?',
    'did you pass ,fail or withdraw?',
    'final result status',
    'certificate category received',
    'at what point did you consider dropping the course?',
    'what best describes your completion behavior?',
    'what was the main reason you dropped out or struggled?',
    'did you fail due to assignment score, exam score, or both?'
]

# ---- Leakage for MARKS model (exam admin info, NOT scores) ----
marks_leakage_cols = [
    'did you meet the minimum assignment score required for certification?',
    'did you register for the proctored final exam?',
    'if registered, did you actually appear for the exam?',
    'reason for not appearing (if applicable)',
    'did you meet the exam minimum score required for certification?'
]

# Combine leakage columns
leakage_cols = dropout_leakage_cols + marks_leakage_cols

# Drop leakage columns safely
df.drop(columns=[col for col in leakage_cols if col in df.columns], inplace=True)



In [7]:
# 7. Convert Numeric-like columns

num_like_cols = [
    'How many lectures did you complete?',
    'How many assignments did you complete?',
    'Average study time per week',
    'Initial motivation level (1–5)',
    'How many total exams you had to take in order to pass the course?'
]

for col in num_like_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')



In [8]:
# 8. Handle Missing Values

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna("unknown")
    else:
        df[col] = df[col].fillna(df[col].median())

In [9]:

# 9. Remove duplicates
df.drop_duplicates(inplace=True)


In [10]:
# 10. Save Cleaned CSV

df.to_csv("cleaned_dataset.csv", index=False)

print(" DATA CLEANING COMPLETED SUCCESSFULLY")


 DATA CLEANING COMPLETED SUCCESSFULLY
